# 01 · From a language model to an application

**Session 01 · Building AI applications for research and data · notebook 1 of 6**

## Objective

A language model writes from two sources: the text in its prompt and what it learned in training. This notebook shows the difference. The same research question is asked with nothing but the question, then with evidence placed in the prompt. The two answers establish the first two components of an application: the instructions, which set the model's behaviour, and the context, which supplies the evidence. The application writes both, and the model reads them.

## The application after this notebook

Diagram: the question and the context feed the prompt, which goes to the model and produces the answer.

The prompt is the only component this notebook builds. Later notebooks add the document store, retrieval, data tools, the agent loop, and the verifier around it.

The whole session uses one research question, so that each method is judged on the same case:

> How have internet access and economic development evolved in Rwanda and Kenya? Compare the recent available data, summarize the main patterns, and cite the sources.

## Contents

1. [Section 1](#scrollTo=section-1) defines the three parts of a prompt and who writes each.
2. [Section 2](#scrollTo=section-2) asks the question with no evidence. Four of the six figures checked are close to the official values, two are off, and none can be traced to a source.
3. [Section 3](#scrollTo=section-3) places four country briefs in the prompt. Every figure now carries a citation.
4. [Section 4](#scrollTo=section-4) asks for a year the brief does not cover. The model says so and gives the closest year.
5. [Section 5](#scrollTo=section-5) asks an ambiguous question. With one added instruction, the model asks which meaning is intended.
6. [Section 6](#scrollTo=section-6) prints the messages as the model received them.
7. [Section 7](#scrollTo=section-7) counts the tokens the context cost and explains why a whole collection cannot be supplied this way.

## How to run it

Run the cells in order. Without an API key, the model's outputs are recorded ones, labelled `[recorded output: ...]`. Each cell names its recorded scenario with `w.use(...)` before it calls the model. With a key in `WORKSHOP_API_KEY`, the same cells call the model live, the answer streams into its bubble as the model writes, and the wording differs from the recorded answers.

The next cell installs the session's tools and loads its helper module. The first run takes about a minute. The cell is setup only.

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/worldbank/ai4data.git"   # the program's repository
BRANCH = "main"   # the branch the Colab setup clones
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook replays recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
model = w.make_model()
print("Model:", w.mode(model))

Model: recorded outputs (no WORKSHOP_API_KEY set)


## The stack

| Need | Library |
|---|---|
| Documents, splitting, embeddings, vector store, prompts, tools, structured output | [LangChain](https://python.langchain.com/) |
| The agent loop | [LangGraph](https://langchain-ai.github.io/langgraph/), through `langchain.agents.create_agent` |
| Tool server and client over the Model Context Protocol | [FastMCP](https://gofastmcp.com/) |
| Number verification | [`pcn-core`](https://github.com/worldbank/pcn), the Python core of the Proof-Carrying Numbers protocol |
| Output schemas | [Pydantic](https://docs.pydantic.dev/) |

`workshop.py` holds what these do not: the worked-example data, a chat model that replays recorded outputs when no API key is configured, and a few printing and evaluation helpers.

The setup cell imported the helper module as `w` and created `model`, a LangChain chat model: a live one when a key is set, otherwise one that replays recorded answers. The helper functions used in this notebook, in the order they appear:

| Call | What it does |
|---|---|
| `w.use(name)` | Names the recorded scenario for the next model call, so that the replay returns the right answer. Has no effect on a live model. |
| `w.show(message, role=...)` | Draws a message as a chat bubble, with the text rendered as Markdown. |
| `w.answer(chain, inputs)` | Calls the chain and draws the answer as a bubble. With a live model the tokens stream into the bubble; without a key the recorded answer is drawn at once. Returns the message. |
| `w.indicator_lookup(indicator, country, start=...)` | Returns the observations of a World Development Indicators series for one country, from the session's data file. |
| `w.latest(result)` | The last observation of such a result. |
| `w.show_table(...)`, `w.show_claim_check(...)`, `w.show_prompt(...)` | Draw a table, the claims comparison, and the prompt as HTML. |
| `w.load_documents()` | Loads the eight session documents as LangChain `Document` objects, with the file's header as metadata. |

## 1. The three parts of a prompt

A call to a chat model sends a list of messages. The first is the system message, which the model treats as standing instructions for the whole exchange. The others are user messages. An application composes three parts into these messages:

- **Instructions**: how the model should behave, written by the application's builders.
- **Context**: the evidence for this question, selected by the application. Here it is documents; later it is passages and data records.
- **Question**: the user's text.

The parts are kept in separate slots because a different person changes each one, at a different time.

The next cell writes three sets of instructions. The basic set only names the role. The grounded set adds the three rules this session depends on: answer from the context, cite the document after each figure, and say when something is missing. The clarifying set adds one more rule, for section 5.

In [2]:
INSTRUCTIONS_BASIC = "You are a research assistant. Answer the question."

INSTRUCTIONS_GROUNDED = """You are a research assistant for development data.
Answer only from the supplied context. Cite the document id in square brackets after each figure.
If the context does not contain what is needed, say so and give the closest available information."""

INSTRUCTIONS_CLARIFY = INSTRUCTIONS_GROUNDED + """
If the question is ambiguous, say which readings are possible and ask which one is meant before answering."""

A prompt template holds the three slots. `ChatPromptTemplate.from_messages` takes a list of `(role, text)` pairs, and each `{name}` in a text is a slot that is filled at call time. The instructions become the system message. The context and the question go together into one user message, in a fixed layout, so that the model can tell them apart. `prompt | model` joins the two steps into one object: the output of the template becomes the input of the model. That object, `chain`, is the whole application at this stage.

In [3]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "{instructions}"),                                   # the system message: the instructions slot
    ("human", "CONTEXT:\n{context}\n\nQUESTION: {question}"),      # one user message: the context and the question slots
])
chain = prompt | model   # fill the template, then send the messages to the model

w.show(w.MAIN_QUESTION, role="user")

Animation: instructions set by the developer, context supplied by the application and a question asked by the user are combined into one prompt, which goes to the model and produces an answer that cites its source.

*The animation loops. Its text comes from this notebook: the instructions are from the cell above, the context and the answer are from section 3, and the question is the worked example. Section 6 shows the same three parts as the model received them.*

## 2. The model without context

Without context, the model answers from training data: text it read before its training ended, compressed into its parameters. It holds no records, only patterns of text, so its figures are reconstructions.

The next cell makes the first call. `w.answer` passes one dictionary with a value for each slot to the chain, and draws the reply. The context slot gets the text `(none)`, so the model sees that no evidence was supplied. `w.use` names the recorded scenario first, so that the replay returns the answer recorded for this exact call.

In [4]:
w.use("nb01_zero_context")
answer = w.answer(chain, {"instructions": INSTRUCTIONS_BASIC, "context": "(none)", "question": w.MAIN_QUESTION})

Indicator,Kenya,Rwanda
Population,~55 million,~14 million
GDP (current US$),~$110–115 billion,~$13–14 billion
GDP per capita,"~$2,000–2,200","~$950–1,050"
Real GDP growth,"~5% (2023), ~4–5% (2024)","~8% (2023), ~9–10% (2024)"
Internet users (share of population),roughly 40%,roughly 30–40% (estimates vary widely)
Mobile broadband,"Large base of mobile data subscriptions, widely used",Expanding; 4G population coverage is high
Mobile money,Very high adoption (M-Pesa is a global reference case),"High adoption, growing via national platforms"
Human Development Index,~0.58 (mid-tier),"~0.53 (low-medium tier, improving)"
Poverty,~36% national poverty line (2020 survey),Fell from ~77% (2001) to ~38% (2016/17); further decline reported in later surveys


The answer cannot be judged without reference values. The next cell prints the official values from the World Development Indicators for the two measures the answer leans on most: the share of people using the internet and GDP per capita. `w.indicator_lookup` returns the observations of one series for one country, and the loop keeps the 2023 and 2024 values.

In [5]:
rows = []
for ind in ["IT.NET.USER.ZS", "NY.GDP.PCAP.CD"]:            # internet users (% of population), GDP per capita (current US$)
    fmt = "{:,.0f}" if ind == "NY.GDP.PCAP.CD" else "{:.1f}"
    for country in ["Kenya", "Rwanda"]:
        r = w.indicator_lookup(ind, country, start=2023)   # observations from 2023 on
        by_year = {o["year"]: fmt.format(o["value"]) for o in r["observations"]}
        rows.append((r["name"], country, by_year[2023], by_year[2024]))
w.show_table(["Indicator", "Country", "2023", "2024"], rows, source="Source: World Bank, World Development Indicators.")

Indicator,Country,2023,2024
Individuals using the Internet (% of population),Kenya,32.1,35.0
Individuals using the Internet (% of population),Rwanda,28.8,31.7
GDP per capita (current US$),Kenya,"1,943","2,133"
GDP per capita (current US$),Rwanda,"1,061","1,060"


The next cell takes six claims from the answer and sets each beside the official value. Each row holds the claim, the model's words, the value from the data, and a verdict. `value` returns the latest observation of a series, and `growth` returns the growth rate of a given year.

In [6]:
def value(ind, country):
    o = w.latest(w.indicator_lookup(ind, country, start=2023))
    return o["value"], o["year"]


def growth(country, year):
    return next(o["value"] for o in w.indicator_lookup("NY.GDP.MKTP.KD.ZG", country, start=year)["observations"] if o["year"] == year)


rows = [  # (claim, what the model said, what the data says, verdict)
    ("Kenya · internet use", "“roughly 40%” of the population", "{:.1f}% of people online in {}".format(*value("IT.NET.USER.ZS", "Kenya")), "above the data"),
    ("Rwanda · internet use", "“roughly 30–40% (estimates vary widely)”", "{:.1f}% of people online in {}".format(*value("IT.NET.USER.ZS", "Rwanda")), "close"),
    ("Kenya · GDP per capita", "“~$2,000–2,200”", "{:,.0f} US dollars in {}".format(*value("NY.GDP.PCAP.CD", "Kenya")), "close"),
    ("Rwanda · GDP per capita", "“~$950–1,050”", "{:,.0f} US dollars in {}".format(*value("NY.GDP.PCAP.CD", "Rwanda")), "close"),
    ("Kenya · GDP growth", "“~5% (2023), ~4–5% (2024)”", f"{growth('Kenya', 2023):.1f} percent in 2023, {growth('Kenya', 2024):.1f} percent in 2024", "close"),
    ("Rwanda · GDP growth", "“~8% (2023), ~9–10% (2024)”", f"{growth('Rwanda', 2023):.1f} percent in 2023, {growth('Rwanda', 2024):.1f} percent in 2024", "2024 above the data"),
]
w.show_claim_check(rows)

Claim,What the model said,What the data says,Verdict,Source named
Kenya · internet use,“roughly 40%” of the population,35.0% of people online in 2024,above the data,none
Rwanda · internet use,“roughly 30–40% (estimates vary widely)”,31.7% of people online in 2024,close,none
Kenya · GDP per capita,"“~$2,000–2,200”","2,133 US dollars in 2024",close,none
Rwanda · GDP per capita,"“~$950–1,050”","1,060 US dollars in 2024",close,none
Kenya · GDP growth,"“~5% (2023), ~4–5% (2024)”","5.7 percent in 2023, 4.7 percent in 2024",close,none
Rwanda · GDP growth,"“~8% (2023), ~9–10% (2024)”","8.6 percent in 2023, 7.2 percent in 2024",2024 above the data,none


Four of the six claims are close to the official values. Two are off: Kenya's internet share, given as roughly 40 percent against 35.0 in 2024, and Rwanda's 2024 growth, given as 9 to 10 percent against 7.2. The answer is hedged, structured, and ends with a reading list, and none of that helps a reader find the two wrong figures. Every figure is a rounded range, most carry no year, and the sources are institutions to consult, so no figure can be traced to a record. A different model or different instructions would change the wording. The figures would still be reconstructions, because the prompt contains no data.

## 3. The same question with context

Context is text that the application places in the prompt for the model to read. Under the grounded instructions, the model answers only from it and cites it.

The next cell builds the context. `w.load_documents` returns the eight session documents, each with an `id` in its metadata, and the cell keeps the four country briefs. Each brief is prefixed with its id in square brackets, which is the form the instructions ask the model to cite. The size is printed as characters and as an estimate of tokens, at four characters per token.

In [7]:
docs = {d.metadata["id"]: d for d in w.load_documents()}
context = "\n\n".join(f"[{i}]\n{docs[i].page_content}" for i in ["kenya-digital", "rwanda-digital", "kenya-economy", "rwanda-economy"])
print(f"{len(context):,} characters of context, roughly {len(context)//4:,} tokens")

4,098 characters of context, roughly 1,024 tokens


The call is the same as in section 2, with the grounded instructions in the first slot and the four briefs in the second.

In [8]:
w.use("nb01_with_context")
answer = w.answer(chain, {"instructions": INSTRUCTIONS_GROUNDED, "context": context, "question": w.MAIN_QUESTION})

Indicator,Kenya,Rwanda
Internet use (% of population),35.0 [kenya-digital],31.7 [rwanda-digital]
Mobile subscriptions (per 100 people),126.5 [kenya-digital],93.2 [rwanda-digital]
GDP per capita (current US$),"2,133 [kenya-economy]","1,060 [rwanda-economy]"
Real GDP growth,4.7% [kenya-economy],7.2% [rwanda-economy]
Population,56.4 million [kenya-digital],14.3 million [rwanda-digital]


Diagram: the same question without context gets a hedged answer of rounded ranges with no citation, two of its figures off; with the four country briefs as context it gets a dated answer that cites its source.

*The two answers, shortened. They are the recorded outputs of sections 2 and 3.*

Every figure now carries a citation to a brief, the comparison is specific, and the model notes two statements in the briefs that it could not verify from the years listed. The application chose the four briefs and put them in the prompt. The model wrote from that text. For the rest of the session, the application decides what evidence the model sees, and the quality of the answer follows from that decision.

## 4. A question the evidence cannot answer

An instruction sets the model's behaviour when the evidence falls short. The grounded instructions say: if the context does not contain what is needed, say so and give the closest available information. The next cell supplies only the Kenya brief, which ends at 2024, and asks for 2025.

In [9]:
w.use("nb01_missing_info")
answer = w.answer(chain, {"instructions": INSTRUCTIONS_GROUNDED, "context": docs["kenya-digital"].page_content,
                          "question": "What share of Kenya's population used the internet in 2025?"})

The model says that 2025 is absent, names 2024 as the latest year, and gives 35.0 percent with a citation. The behaviour came from one sentence in the instructions.

## 5. An ambiguous question under two sets of instructions

"How connected is Kenya?" has at least three readings: internet use, mobile subscriptions, or broadband coverage. A model picks a reading unless it is told to ask. The next cell asks the question twice with the same context: first with the basic instructions, then with the clarifying set, whose added rule tells the model to list the readings and ask which one is meant.

In [10]:
print("Without an instruction to clarify:\n")
w.use("nb01_ambiguous_no_instruction")
w.answer(chain, {"instructions": INSTRUCTIONS_BASIC, "context": docs["kenya-digital"].page_content, "question": "How connected is Kenya?"})

print("\n" + "=" * 100 + "\nWith the instruction:\n")
w.use("nb01_ambiguous_with_instruction")
w.answer(chain, {"instructions": INSTRUCTIONS_CLARIFY, "context": docs["kenya-digital"].page_content, "question": "How connected is Kenya?"})

Without an instruction to clarify:




With the instruction:



AIMessage(content='"Connected" could mean a few different things here, so which do you mean?\n\n1. **Internet use:** the share of the population using the internet. This is the narrower measure of online access.\n2. **Mobile subscriptions:** mobile cellular subscriptions per 100 people. This measures subscriptions, not how many people have a phone, since the figure can exceed 100 and counts inactive SIMs.\n3. **Both together:** a comparison of the two, which the context suggests is a gap where mobile coverage has grown faster than internet use.\n\nWhich one would you like me to answer? If you want the comparison, I can give the 2010 to 2024 figures for both indicators.', additional_kwargs={}, response_metadata={'recorded': 'nb01_ambiguous_with_instruction'}, id='lc_run--01a16190-3c17-7bd2-8474-0837c56004d5-0', tool_calls=[], invalid_tool_calls=[])

With the basic instructions, the model answers at length, covers two readings, and adds a growth analysis the question did not ask for. With the clarifying instruction, it names three readings and asks. An answer to the model's own reading looks as confident as an answer to the user's reading, so the user cannot tell them apart. For a research application, asking costs one exchange and removes that risk.

## 6. The prompt as sent to the model

`prompt.format_messages` fills the template with values and returns the message list without calling the model. The next cell does this for the last call of section 5 and draws the messages in the colours of section 1. The instructions arrive as the system message. The context and the question arrive together in one user message. This is everything the model sees.

In [11]:
messages = prompt.format_messages(instructions=INSTRUCTIONS_CLARIFY, context=docs["kenya-digital"].page_content, question="How connected is Kenya?")
w.show_prompt(messages)

## 7. The cost of context

A model reads a limited amount of text per call, called its context window and measured in tokens. A token is a word or a part of a word, about four characters of English on average. The four briefs took about a thousand tokens. A collection of thousands of documents exceeds any window, the price of a call grows with its tokens, and the passages that answer one question are a small part of the whole. An application therefore selects, for each question, the part of the collection that the model should read. Notebooks 02 and 03 build that selection.

Diagram: the four briefs of this notebook, about 1,024 tokens, fit inside the model's context window, while a real collection of thousands of documents cannot, so the application must choose what the model reads.

*The token count is the one printed in section 3.*

## Key points

- A prompt has three parts: instructions, context, and the question. A different person writes each, so they are kept in separate slots.
- Without context, the figures are reconstructions from training data. Four of the six checked were close, two were off, and none could be traced to a record.
- With the briefs as context, every figure cited a brief. The application chose the evidence, and the model wrote from it.
- Instructions set behaviour: cite, say when information is missing, ask when a question is ambiguous. They cannot supply data.
- The context window limits what one call can read, so the application has to select the evidence. The next two notebooks build the selection.

## Exercise

1. Read the answers of sections 2 and 3 side by side. For each number, note whether it can be checked and where. Then decide which answer could be sent to a colleague, and give the reason.
2. In the first cell of section 1, add a sentence to `INSTRUCTIONS_GROUNDED`, for example "Answer in no more than three sentences." Run that cell again, then run the cell of section 6. The Instructions block now shows the added sentence exactly as the model would receive it. The recorded answers do not react to new instructions. With a live key in `WORKSHOP_API_KEY`, rerun section 3 to see how the answer changes.

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows. See also the [paper](https://arxiv.org/abs/2509.06902) and the [`pcn` repository](https://github.com/worldbank/pcn).
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.

**Next:** [02 · Preparing knowledge for an application](https://colab.research.google.com/github/worldbank/ai4data/blob/main/training/session-01-building-ai-applications/notebooks/02-preparing-knowledge.ipynb)